# StormEngine V6 — Frozen-checkpoint evaluation

Use this notebook only after an experiment configuration has been frozen. Model selection uses validation data; the held-out 2017 test set is evaluated separately.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
ARTIFACT_DIR = REPO / 'artifacts' / 'v6_2010_2017'
CHECKPOINT = ARTIFACT_DIR / 'best.pt'
EVALUATION_DIR = ARTIFACT_DIR / 'evaluation_test'
print('Repository:', REPO)
print('Checkpoint:', CHECKPOINT)

## 1. Run the held-out test exactly once

Leave `RUN_FINAL_TEST=False` during model development. Set it to `True` only after the architecture and hyperparameters are fixed.

In [ ]:
RUN_FINAL_TEST = False
DEVICE = 'cuda'  # use 'cpu' only for a small local check
LOCAL_CONFIG = REPO / 'configs' / 'era5_2010_2017_windows.local.yaml'
if RUN_FINAL_TEST:
    assert CHECKPOINT.exists(), CHECKPOINT
    command = [
        sys.executable, '-u', str(REPO / 'scripts' / 'evaluate.py'),
        '--config', str(LOCAL_CONFIG), '--checkpoint', str(CHECKPOINT),
        '--split', 'test', '--device', DEVICE, '--save-examples', '3'
    ]
    subprocess.run(command, cwd=REPO, check=True)
else:
    print('Final test is locked. Validation results may be inspected without changing this flag.')

## 2. Training and validation curves

In [ ]:
history = json.loads((ARTIFACT_DIR / 'history.json').read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [row['train_loss'] for row in history], label='train')
plt.plot(epochs, [row['validation_loss'] for row in history], label='validation')
plt.xlabel('Epoch'); plt.ylabel('Sea-weighted normalized MSE')
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()
best = min(history, key=lambda row: row['validation_loss'])
print('Best validation epoch:', best)

## 3. Aggregate and 1–6 hour metrics

In [ ]:
METRICS_PATH = EVALUATION_DIR / 'metrics_by_lead.json'
assert METRICS_PATH.exists(), 'Run the frozen evaluation first: ' + str(METRICS_PATH)
evaluation = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
aggregate = evaluation['metrics']['aggregate']
for region in ('full', 'land', 'sea'):
    print('\n', region.upper())
    for variable, values in aggregate[region].items():
        print(f"  {variable:5s} MAE={values['mae']:.4f} RMSE={values['rmse']:.4f}")

In [ ]:
by_lead = evaluation['metrics']['by_lead_hour']
variables = evaluation['variables']
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for variable, axis in zip(variables, axes.flat):
    hours = sorted(map(int, by_lead))
    for region in ('full', 'land', 'sea'):
        values = [by_lead[str(hour)][region][variable]['rmse'] for hour in hours]
        axis.plot(hours, values, marker='o', label=region)
    axis.set_title(variable); axis.set_xlabel('Lead hour'); axis.set_ylabel('RMSE'); axis.grid(alpha=.3)
axes.flat[-1].axis('off')
axes.flat[0].legend(); fig.tight_layout(); plt.show()

## 4. Forecast, ERA5 target, and error map

In [ ]:
example_paths = [EVALUATION_DIR / path for path in evaluation['examples']]
assert example_paths, 'Evaluation did not export examples'
example = np.load(example_paths[0])
VARIABLE = 't2m'
LEAD_HOUR = 6
names = example['variables'].astype(str).tolist()
channel, lead = names.index(VARIABLE), LEAD_HOUR - 1
prediction = example['prediction'][lead, channel]
target = example['target'][lead, channel]
error = prediction - target
extent = [example['longitudes'].min(), example['longitudes'].max(), example['latitudes'].min(), example['latitudes'].max()]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, field, title, cmap in zip(axes, (target, prediction, error), ('ERA5 target', 'StormEngine forecast', 'Forecast error'), ('viridis', 'viridis', 'RdBu_r')):
    image = axis.imshow(field, origin='lower', extent=extent, aspect='auto', cmap=cmap)
    axis.set_title(f'{title}: {VARIABLE} +{LEAD_HOUR}h'); axis.set_xlabel('Longitude'); axis.set_ylabel('Latitude')
    fig.colorbar(image, ax=axis, shrink=.8)
fig.tight_layout(); plt.show()
print('Valid time:', example['forecast_times'][lead])